# NB_hospital_daily_summary
Builds `<catalog>.gold.hospital_daily_summary` per STTM `sttm_silver_to_gold.xlsx` (sheet `hospital_daily_summary`).

* **Grain:** one row per hospital per calendar day (days with lab tests)
* **Driving table:** `silver.lab_results` grouped by hospital_id, test_date
* **Joins (LEFT):** `silver.hospitals` (hospital_id), `silver.calendar` (date_key = yyyyMMdd of test_date)
* **Window:** `LAG(total_tests)` partitioned by hospital_id ordered by summary_date (day-over-day trend)
* **Filters:** none
* **Load type:** FULL — `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
import re

dbutils.widgets.text("catalog", "adb_caresync")
catalog = dbutils.widgets.get("catalog").strip()
if not re.fullmatch(r"[A-Za-z0-9_]+", catalog):
    raise ValueError(f"Invalid catalog name: {catalog!r}")

src = f"{catalog}.silver"
target_table = f"{catalog}.gold.hospital_daily_summary"
print("Target:", target_table)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH daily AS (
    SELECT
        hospital_id,
        test_date                             AS summary_date,
        COUNT(*)                              AS total_tests,
        COUNT(DISTINCT patient_id)            AS unique_patients_tested,
        COUNT(DISTINCT doctor_id)             AS active_doctor_count,
        COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical')  AS critical_test_count
    FROM {src}.lab_results
    GROUP BY hospital_id, test_date
),
with_lag AS (
    SELECT
        d.*,
        LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
    FROM daily d
)
SELECT
    CAST(w.hospital_id AS INT)                                             AS hospital_id,
    CAST(h.hospital_name AS STRING)                                        AS hospital_name,
    CAST(w.summary_date AS DATE)                                           AS summary_date,
    CAST(c.day_name AS STRING)                                             AS day_name,
    CAST(c.is_weekend AS BOOLEAN)                                          AS is_weekend,
    CAST(w.total_tests AS INT)                                             AS total_tests,
    CAST(w.unique_patients_tested AS INT)                                  AS unique_patients_tested,
    CAST(w.active_doctor_count AS INT)                                     AS active_doctor_count,
    CAST(w.abnormal_test_count AS INT)                                     AS abnormal_test_count,
    CAST(w.critical_test_count AS INT)                                     AS critical_test_count,
    CAST(ROUND(w.abnormal_test_count / w.total_tests * 100, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(w.critical_test_count / w.total_tests * 100, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
    CAST(w.prev_day_total_tests AS INT)                                    AS prev_day_total_tests,
    CAST(
        CASE WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
             ELSE ROUND((w.total_tests - w.prev_day_total_tests) / w.prev_day_total_tests * 100, 2)
        END AS DECIMAL(6,2))                                               AS test_volume_change_pct
FROM with_lag w
LEFT JOIN {src}.hospitals h
       ON w.hospital_id = h.hospital_id
LEFT JOIN {src}.calendar c
       ON c.date_key = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT)
""")

In [0]:
# tgt = spark.table(target_table)
# tgt_cnt = tgt.count()
# expected = spark.table(f"{src}.lab_results").select("hospital_id", "test_date").distinct().count()
# dup_cnt = tgt_cnt - tgt.select("hospital_id", "summary_date").distinct().count()
# src_tests = spark.table(f"{src}.lab_results").count()
# tgt_tests = tgt.agg({"total_tests": "sum"}).collect()[0][0]

# print(f"rows: {tgt_cnt} (expected {expected}) | duplicate grain: {dup_cnt} | tests: {tgt_tests} (source {src_tests})")
# assert tgt_cnt == expected, "Row count mismatch vs distinct hospital/day in lab_results"
# assert dup_cnt == 0, "Grain violated: duplicate hospital_id + summary_date"
# assert tgt_tests == src_tests, "Total tests do not reconcile with silver.lab_results"

# display(tgt.orderBy("hospital_id", "summary_date").limit(10))

In [0]:
updated_df = spark.table(target_table)
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))